# Phase 1 Diagnostic — Railgun Cooperation Failure

Goal: prove the cooperation failure mode exists and is structural, before any algorithm work.

Outline:
1. Setup (env, drive, clone Railgun)
2. Reproduce vanilla Railgun
3. LaCAM expert trajectory generation
4. Build diagnostic dataset
5. Core analyses (5 questions)
6. Negative-result setup (Song et al. shaping)
7. Findings document

## Setup

In [ ]:
# Cell 1 — Environment
!pip install pogema
!pip install torch torchvision
!pip install networkx matplotlib seaborn
!pip install numpy pandas tqdm scikit-learn pyarrow

In [ ]:
# Cell 2 — Mount Drive (persistent storage)
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/railgun_followup'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/diagnostic_data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/figures', exist_ok=True)

In [ ]:
# Cell 3 — Clone Railgun repo (after Yimin gives access)
!git clone <railgun_repo_url> /content/railgun
%cd /content/railgun
!pip install -e .

## Section 1 — Reproduce vanilla Railgun

In [ ]:
# Cell 4 — Load pretrained Railgun
import torch
from railgun.model import RailgunUNet  # adjust import based on actual repo structure

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = RailgunUNet()
model.load_state_dict(torch.load('checkpoints/railgun_pretrained.pt'))
model.eval().to(device)

In [ ]:
# Cell 5 — Single scenario sanity check
from pogema import pogema_v0, GridConfig

grid_config = GridConfig(
    map_name='warehouse',
    num_agents=64,
    max_episode_steps=256,
    seed=42,
)
env = pogema_v0(grid_config=grid_config)

obs, info = env.reset()
print(f'Map shape: {env.grid.shape}')
print(f'Number of agents: {len(obs)}')

In [ ]:
# Cell 6 — Confirm inference pipeline
# Run Railgun for one timestep, log Fout, sample actions, step env.
# Expected: actions execute, agents move, no errors.
raise NotImplementedError('fill in once Railgun forward signature is known')

## Section 2 — LaCAM expert trajectory generation

In [ ]:
# Cell 7 — Run LaCAM on the same scenario
from railgun.experts import LaCAMSolver  # adjust based on repo

solver = LaCAMSolver()
expert_trajectory = solver.solve(env)

In [ ]:
# Cell 8 — Verify expert trajectory (no collisions, all agents reach goals)
assert verify_trajectory(expert_trajectory, env)

### Cell 9 — Align Railgun and LaCAM step-by-step

Use **teacher forcing**: feed LaCAM's state sequence into Railgun, log Railgun's predictions at each LaCAM-induced state. This isolates per-cell prediction error from cascading rollout error.

## Section 3 — Build diagnostic dataset

In [ ]:
# Cell 10 — Logging schema (per (scenario, timestep, cell))
import numpy as np

diagnostic_record_schema = {
    'scenario_id': str,
    'timestep': int,
    'cell_i': int,
    'cell_j': int,
    'agent_present': bool,
    'agent_id': 'int or None',
    'railgun_action_dist': 'np.array(5)',
    'railgun_argmax': int,
    'lacam_action': int,
    'disagreement': bool,
    'local_agent_density': float,
    'graph_betweenness': float,
    'path_intersection_count': int,
    'lacam_revisit_count': int,
}

In [ ]:
# Cell 11 — Diagnostic generation loop
from tqdm import tqdm
import pandas as pd

agent_counts = [64, 96, 128, 160, 192]
seeds = list(range(50))  # 250 total scenarios
records = []

for n_agents in agent_counts:
    for seed in tqdm(seeds, desc=f'{n_agents} agents'):
        try:
            scenario_records = run_diagnostic_scenario(
                map_name='warehouse',
                num_agents=n_agents,
                seed=seed,
                model=model,
                solver=solver,
            )
            records.extend(scenario_records)
        except Exception as e:
            print(f'Failed: {n_agents} agents, seed {seed}: {e}')

df = pd.DataFrame(records)
df.to_parquet(f'{PROJECT_DIR}/diagnostic_data/warehouse_diagnostic.parquet')
print(f'Generated {len(df)} records across {len(seeds) * len(agent_counts)} scenarios')

In [ ]:
# Cell 12 — Sanity check dataset
print(df.describe())
print(f'Disagreement rate: {df.disagreement.mean():.3f}')
print('Disagreement rate by scenario:')
print(df.groupby('scenario_id').apply(
    lambda g: g[g.agent_present].disagreement.mean()
))

In [ ]:
# Cell 13 — Save intermediate artifacts
df.to_parquet(f'{PROJECT_DIR}/diagnostic_data/warehouse_full.parquet')

## Section 4 — Core diagnostic analysis

In [ ]:
# Cell 14 — Q1: Does disagreement correlate with criticality?
import seaborn as sns
import matplotlib.pyplot as plt

df_agents = df[df.agent_present]

criticality_metrics = [
    'local_agent_density',
    'graph_betweenness',
    'path_intersection_count',
    'lacam_revisit_count',
]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, metric in zip(axes, criticality_metrics):
    sns.boxplot(data=df_agents, x='disagreement', y=metric, ax=ax)
    ax.set_title(f'{metric}\n(by disagreement)')
plt.savefig(f'{PROJECT_DIR}/figures/criticality_vs_disagreement.png', dpi=150)
plt.show()

In [ ]:
# Cell 15 — Q2: Per-cell loss vs criticality
import torch.nn.functional as F

def compute_per_cell_loss(railgun_dist, expert_action):
    return F.cross_entropy(
        torch.tensor(railgun_dist).unsqueeze(0),
        torch.tensor([expert_action]),
    ).item()

df_agents['per_cell_loss'] = df_agents.apply(
    lambda r: compute_per_cell_loss(r.railgun_action_dist, r.lacam_action),
    axis=1,
)

correlations = {
    metric: df_agents[['per_cell_loss', metric]].corr().iloc[0, 1]
    for metric in criticality_metrics
}
print('Pearson correlation: per-cell loss vs criticality metric')
for metric, corr in correlations.items():
    print(f'  {metric}: {corr:.4f}')

In [ ]:
# Cell 16 — Q3: Failure rate vs critical-cell density
scenario_summary = df.groupby('scenario_id').agg(
    n_agents=('agent_id', 'nunique'),
    critical_fraction=('local_agent_density', lambda x: (x > x.quantile(0.9)).mean()),
    csr=('agent_at_goal', 'all'),  # adjust based on actual logging
)

sns.scatterplot(
    data=scenario_summary,
    x='critical_fraction',
    y='csr',
    hue='n_agents',
    palette='viridis',
)
plt.savefig(f'{PROJECT_DIR}/figures/critical_fraction_vs_csr.png', dpi=150)
plt.show()

In [ ]:
# Cell 17 — Q4: Spatial distribution of failures
warehouse_map = env.grid
disagreement_heatmap = np.zeros_like(warehouse_map, dtype=float)
counts = np.zeros_like(warehouse_map, dtype=int)

for _, row in df_agents.iterrows():
    disagreement_heatmap[row.cell_i, row.cell_j] += float(row.disagreement)
    counts[row.cell_i, row.cell_j] += 1

disagreement_rate = disagreement_heatmap / np.maximum(counts, 1)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].imshow(warehouse_map, cmap='gray')
axes[0].set_title('Warehouse map')
axes[1].imshow(disagreement_rate, cmap='hot')
axes[1].set_title('Disagreement rate by cell')
plt.savefig(f'{PROJECT_DIR}/figures/disagreement_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Cell 18 — Q5: Best criticality metric (logistic regression AUC)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

results = {}
for metric in criticality_metrics:
    X = df_agents[[metric]].values
    y = df_agents['disagreement'].astype(int).values
    lr = LogisticRegression()
    lr.fit(X, y)
    auc = roc_auc_score(y, lr.predict_proba(X)[:, 1])
    results[metric] = auc

print('AUC for predicting disagreement from criticality metric:')
for metric, auc in sorted(results.items(), key=lambda x: -x[1]):
    print(f'  {metric}: {auc:.4f}')

## Section 5 — Negative-result setup (Song et al. cooperative shaping)

In [ ]:
# Cell 19 — Adapt Song et al. cooperative shaping for Railgun
def cooperative_reward_signal(state, action, next_state, agent_id, alpha=0.5):
    """Own progress + alpha * neighbor progress."""
    own_progress = compute_progress(state, next_state, agent_id)
    neighbor_progress = compute_neighbor_progress(state, next_state, agent_id)
    return own_progress + alpha * neighbor_progress

# Then RL fine-tune Railgun with PPO using this reward.

In [ ]:
# Cell 20 — Compare on warehouse 128, 160 agents:
#   1. Vanilla Railgun
#   2. Railgun + Song et al. shaping (RL fine-tuned)
# Expected: minimal improvement on warehouse collapse — confirms decentralized
# shaping doesn't fix centralized supervised cooperation failures.
raise NotImplementedError

## Section 6 — Findings document

In [ ]:
# Cell 21 — Generate findings
from datetime import datetime

findings = f"""
# Phase 1 Diagnostic Findings — {datetime.now():%Y-%m-%d}

## Summary statistics
- Total scenarios: {df.scenario_id.nunique()}
- Total per-cell records: {len(df)}
- Agent-occupied records: {df.agent_present.sum()}
- Overall disagreement rate: {df_agents.disagreement.mean():.3f}

## Disagreement scaling with agent count
{df_agents.groupby('n_agents').disagreement.mean().to_string()}

## Best criticality metric
{max(results.items(), key=lambda x: x[1])}

## Correlation: per-cell loss vs criticality
{correlations}

## Hypothesis support
- Disagreement rate increases with agent count: [YES/NO]
- Disagreement concentrates in critical cells: [YES/NO]
- Per-cell loss correlates with criticality: [WEAKLY/STRONGLY]
- Spatial clustering at junctions: [YES/NO]

## Decision
- If hypothesis supported: proceed to Phase 2 (negative result)
- If not: pivot to alternative framing
"""

with open(f'{PROJECT_DIR}/diagnostic_findings.md', 'w') as f:
    f.write(findings)
print(findings)